# CLIP 실습: 이미지와 텍스트 임베딩을 같은 공간에서 비교하기

cosine similarity와 contrastive learning의 계산 흐름을 작은 행렬로 이해합니다.


In [ ]:
# numpy는 벡터 계산, matplotlib은 유사도 행렬 시각화에 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 임베딩 예제를 재현하기 위해 시드를 고정합니다.
np.random.seed(66)


In [ ]:
# 장난감 클래스와 텍스트 프롬프트를 정의합니다.
labels = ["cat", "dog", "car", "tree"]
prompts = [f"a photo of a {label}" for label in labels]
dim = 6
base = np.random.normal(0, 1, (len(labels), dim))
image_embeddings = base + np.random.normal(0, 0.08, base.shape)
text_embeddings = base + np.random.normal(0, 0.08, base.shape)

def l2_normalize(x):
    # 코사인 유사도를 쓰기 위해 벡터 길이를 1로 맞춥니다.
    return x / np.linalg.norm(x, axis=1, keepdims=True)

image_embeddings = l2_normalize(image_embeddings)
text_embeddings = l2_normalize(text_embeddings)
similarity = image_embeddings @ text_embeddings.T
print("프롬프트:", prompts)
print("유사도 행렬:\n", np.round(similarity, 2))


In [ ]:
# 유사도 행렬의 대각선이 맞는 이미지-텍스트 쌍입니다.
plt.imshow(similarity, cmap="magma", vmin=-1, vmax=1)
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(prompts)), labels, rotation=45)
plt.yticks(range(len(labels)), labels)
plt.title("image-text similarity")
plt.xlabel("text prompt")
plt.ylabel("image")
plt.show()


In [ ]:
# CLIP의 대조 학습 손실을 아주 작게 구현합니다.
def softmax(x, axis=-1):
    # 수치 안정성을 위해 최댓값을 빼고 softmax를 계산합니다.
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=axis, keepdims=True)

def cross_entropy(prob, target_indices):
    # 정답 위치의 확률이 높을수록 손실이 작아집니다.
    eps = 1e-12
    return -np.mean(np.log(prob[np.arange(len(target_indices)), target_indices] + eps))

temperature = 0.07
logits = similarity / temperature
targets = np.arange(len(labels))
image_to_text_prob = softmax(logits, axis=1)
text_to_image_prob = softmax(logits.T, axis=1)
loss_i2t = cross_entropy(image_to_text_prob, targets)
loss_t2i = cross_entropy(text_to_image_prob, targets)
clip_loss = (loss_i2t + loss_t2i) / 2
print("image->text loss:", round(loss_i2t, 4))
print("text->image loss:", round(loss_t2i, 4))
print("CLIP-style loss:", round(clip_loss, 4))


In [ ]:
# zero-shot 분류 직관: 새 이미지 임베딩이 어떤 텍스트 프롬프트와 가장 가까운지 고릅니다.
new_image = image_embeddings[1] + np.random.normal(0, 0.05, dim)
new_image = new_image / np.linalg.norm(new_image)
scores = new_image @ text_embeddings.T
predicted = labels[int(np.argmax(scores))]
for label, score in zip(labels, scores):
    print(f"{label:>4} score = {score:.3f}")
print("예측:", predicted)


## 관찰 포인트

CLIP은 맞는 이미지-텍스트 쌍의 유사도를 높이고, 배치 안의 다른 쌍과는 멀어지게 학습합니다.
